# 04. 회귀용 데이터 병합 → `09_회귀분석_격자통합데이터.csv`

**원본**: `Analisis3/004_regression_…ipynb`의 앞부분

- Y와 모든 X를 셀 ID로 합쳐 파일 하나로 만든다. **이후 모든 회귀(1_05, analysis)는 이 파일만 읽는다.**
- 원본은 `walk.columns[19]`처럼 컬럼을 **위치**로 골랐다. 파일 컬럼 순서가 바뀌면 조용히 다른 변수가 들어가므로, 정리본은 **이름**으로 고른다.
- 보행시간·IC까지 도로거리는 Analisis2 결과를 그대로 쓴다 (생성 코드는 전달본에 없음).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

In [2]:
# (파일, 원본 컬럼명, 짧은 이름)
SOURCES = [
    (DEPENDENT_PATH, "2024년 07~09시 승차인원", "board_2024"),
    (DEPENDENT_PATH, "2025년 07~09시 승차인원", "board_2025"),
    (DEPENDENT_PATH, "07~09시 승차인원 증감", "change"),
    (DEPENDENT_PATH, "감소 여부", "decreased"),
    (DEPENDENT_PATH, "2024년 종일 승차인원", "board_2024_all"),
    (DEPENDENT_PATH, "2025년 종일 승차인원", "board_2025_all"),
    (DEPENDENT_PATH, "종일 승차인원 증감", "change_all"),
    (WALK_PATH, "ORS GET 보행시간(분)", "walk_min"),
    (WALK_PATH, "최근접 GTX-A역", "nearest_station"),
    (IC_PATH, "IC 접근성", "ic_access"),
    (POPULATION_PATH, "20~50대 인구", "pop_20_50"),
    (POPULATION_PATH, "고령인구", "pop_elderly"),
    (POPULATION_PATH, "시군", "city"),
    (BUILDING_PATH, "건축물 수", "buildings"),
]
model = None
for path, source_column, name in SOURCES:
    part = read_csv(path)[["셀 ID", source_column]].rename(columns={source_column: name})
    part["셀 ID"] = pd.to_numeric(part["셀 ID"], errors="coerce").astype("Int64")
    part = part.dropna(subset=["셀 ID"]).drop_duplicates("셀 ID")
    model = part if model is None else model.merge(part, on="셀 ID", how="left")

model.to_csv(MODEL_DATA_PATH, index=False, encoding="utf-8-sig")
print(f"{len(model)}행 × {model.shape[1]}열 | 결측: {int(model.isna().sum().sum())}개 | 저장: {MODEL_DATA_PATH.name}")
model.head()

132행 × 17열 | 결측: 0개 | 저장: 09_회귀분석_격자통합데이터.csv


,셀 ID,board_2024,board_2025,change,decreased,board_2024_all,board_2025_all,change_all,walk_min,nearest_station,ic_access,competing_share,pop_20_50,pop_elderly,city,buildings,competing_share_missing
0,44,70.0,35.0,-35.0,1,269,164,-105,51.650000,대곡,5.071,24.285714,937.0,276.0,고양,18.0,0
1,76,0.0,2.0,2.0,0,17,35,18,160.710000,운정중앙,1.921,0.000000,33.0,19.0,파주,37.0,1
2,559,11.0,21.0,10.0,0,82,90,8,50.225000,운정중앙,0.319,0.000000,0.0,0.0,파주,46.0,0
3,619,1.0,0.0,-1.0,1,1,4,3,166.555000,운정중앙,9.029,100.000000,0.0,0.0,파주,7.0,0
4,765,12.0,13.0,1.0,0,18,25,7,128.416667,대곡,4.323,41.666667,139.0,78.0,고양,92.0,0


| 컬럼 | 뜻 |
|---|---|
| `board_2024`, `board_2025`, `change`, `decreased` | 07~09시 승차(2024, 2025), 증감, 감소 여부(1=감소) |
| `board_2024_all`, `board_2025_all`, `change_all` | 종일 승차, 종일 증감 |
| `walk_min`, `nearest_station` | 최근접 GTX-A역까지 보행시간(분), 그 역 이름 |
| `ic_access` | **IC까지 도로거리(km)**: 격자에서 가장 가까운 IC까지의 도로 이동거리. 값이 작을수록 IC에 가깝다. 원본 컬럼명은 'IC 접근성'이지만 실제로는 거리다 |
| `competing_share` | 최종모형에서 셀×노선 단위 파일을 별도로 결합하므로 이 격자 통합자료에는 포함하지 않음 |
| `pop_20_50`, `pop_elderly`, `buildings`, `city` | 20~50대 인구, 고령인구, 건축물 수, 고양/파주 |